In [18]:
!pip install kafka-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.1/326.1 kB 2.4 MB/s eta 0:00:00a 0:00:01


In [2]:
import json
import pandas as pd
from kafka import KafkaProducer

In [3]:
producer = KafkaProducer(
bootstrap_servers="kafka:29092",
value_serializer=lambda v: json.dumps(v).encode("utf-8"),
key_serializer=lambda v: v.encode('utf-8') if v else None,
)

print('Producer ready')

In [12]:
df = pd.read_csv('/home/jovyan/data/olist_orders_dataset.csv')
print("Rows to publish:", len(df))

Rows to publish: 99441


In [13]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [14]:
# ==============================
# CEK DATA SEBELUM TRANSFORMASI
# ==============================

# 1. Ukuran data
print("=== Shape ===")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

# 2. Tipe data
print("\n=== Data Types ===")
print(df.dtypes)

# 3. Jumlah null per kolom
print("\n=== Null Count ===")
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_summary = pd.DataFrame({"null_count": null_counts, "null_%": null_pct})
print(null_summary[null_summary["null_count"] > 0])  # hanya tampilkan yang ada null

# 4. Duplikat
print("\n=== Duplicates ===")
print(f"Duplicate rows     : {df.duplicated().sum():,}")
print(f"Duplicate order_id : {df['order_id'].duplicated().sum():,}")

# 5. Nilai unik kolom kategorikal
print("\n=== Unique Values - order_status ===")
print(df["order_status"].value_counts())

# 6. Sample format kolom datetime (cek apakah formatnya konsisten)
print("\n=== Sample Datetime Formats ===")
datetime_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in datetime_cols:
    sample = df[col].dropna().head(3).tolist()
    print(f"{col}:\n  {sample}\n")

# 7. Preview data
print("=== Preview (head 3) ===")
df.head(3)

=== Shape ===
Rows: 99,441 | Columns: 8

=== Data Types ===
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

=== Null Count ===
                               null_count  null_%
order_approved_at                     160    0.16
order_delivered_carrier_date         1783    1.79
order_delivered_customer_date        2965    2.98

=== Duplicates ===
Duplicate rows     : 0
Duplicate order_id : 0

=== Unique Values - order_status ===
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

=== Sample Datetime Formats ===
order_purchase_timestamp:
  ['2017-10-02 1

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [15]:
# -----------------------------
# Transformasi tipe data
# -----------------------------

# Kolom datetime (yang tidak boleh null)
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
df["order_estimated_delivery_date"] = pd.to_datetime(df["order_estimated_delivery_date"])

# Kolom datetime yang boleh null
nullable_datetime_cols = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
]
for col in nullable_datetime_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")  # NaN tetap NaN, tidak error

# Verifikasi hasil
print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Null counts:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [16]:
# -----------------------------
# Publish to Kafka in batch
# -----------------------------
def safe_datetime(val):
    """Konversi datetime ke ISO string, return None jika NaT/null"""
    return val.isoformat() if pd.notna(val) else None

for _, row in df.iterrows():
    msg = {
        "order_id": row["order_id"],
        "customer_id": row["customer_id"],
        "order_status": row["order_status"],
        "order_purchase_timestamp": safe_datetime(row["order_purchase_timestamp"]),
        "order_approved_at": safe_datetime(row["order_approved_at"]),
        "order_delivered_carrier_date": safe_datetime(row["order_delivered_carrier_date"]),
        "order_delivered_customer_date": safe_datetime(row["order_delivered_customer_date"]),
        "order_estimated_delivery_date": safe_datetime(row["order_estimated_delivery_date"]),
    }
    producer.send("orders", msg)

producer.flush()
print("✅Batch publish to Kafka finished")

✅Batch publish to Kafka finished


# 1. Start Spark

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('olist-kafka-producer')
    .master('local[*]')
    .config(
        'spark.jars.packages',
        'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,org.postgresql:postgresql:42.7.3'
    )
    .getOrCreate()
)

spark

In [4]:
import os

DATA_PATH = os.getenv('DATA_PATH', '/home/jovyan/data')
CHECKPOINT_PATH = f"{DATA_PATH}/checkpoint"

print('DATA_PATH =', DATA_PATH)
print('CHECKPOINT_PATH =', CHECKPOINT_PATH)

DATA_PATH = /home/jovyan/work/data
CHECKPOINT_PATH = /home/jovyan/work/data/checkpoint


# 2. Load data

In [14]:
orders_df = spark.read.csv(
    f"{DATA_PATH}/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

orders_df.printSchema()
orders_df.show(5, truncate=False)

print("Rows to publish:", orders_df.count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------

# 3. Transformasi data

In [15]:
from pyspark.sql import functions as F

orders_clean_df = (
    orders_df
    .select(
        F.col('order_id').cast('string'),
        F.col('customer_id').cast('string'),
        F.col('order_status').cast('string'),
        F.to_timestamp('order_purchase_timestamp').alias('order_purchase_timestamp'),
        F.to_timestamp('order_approved_at').alias('order_approved_at'),
        F.to_timestamp('order_delivered_carrier_date').alias('order_delivered_carrier_date'),
        F.to_timestamp('order_delivered_customer_date').alias('order_delivered_customer_date'),
        F.to_timestamp('order_estimated_delivery_date').alias('order_estimated_delivery_date'),
    )
)

orders_clean_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



# 4. Validate

In [16]:

orders_valid_df = (
    orders_clean_df
    .filter(F.col('order_id').isNotNull())
    .filter(F.col('customer_id').isNotNull())
    .dropDuplicates(['order_id'])
)

print('raw orders count:', orders_df.count())
print('valid deduplicated orders count:', orders_valid_df.count())

raw orders count: 99441
valid deduplicated orders count: 99441


# 5. Order by purchase timesstamp

In [ ]:
orders_kafka_df = (
    orders_valid_df
    .orderBy(F.col('order_purchase_timestamp').asc_nulls_last())
)

orders_kafka_df.select(
    'order_id',
    'customer_id',
    'order_status',
    'order_purchase_timestamp'
).show(5, truncate=False)

In [1]:
import json
from kafka import KafkaProducer

producer = KafkaProducer(
bootstrap_servers="kafka:29092",
value_serializer=lambda v: json.dumps(v).encode("utf-8"),
key_serializer=lambda v: v.encode('utf-8') if v else None,
)

print('Producer ready')

Producer ready
